In [2]:
%cd /content/drive/MyDrive/Sentiment_classification

/content/drive/MyDrive/Sentiment_classification


In [ ]:
import importlib
from src import preprocess , encoded_tokens, pretrained_embeddings,models
importlib.reload(preprocess)
importlib.reload(encoded_tokens)
importlib.reload(pretrained_embeddings)
importlib.reload(models)
from src.preprocess import load_and_preprocess
from src.encoded_tokens import build_vocab, text_to_sequence, pad_sequence
from src.pretrained_embeddings import load_w2v, load_glove
from src.models import CNNModel

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import time
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
df_path = "/content/drive/MyDrive/Sentiment_classification/data/IMDB Dataset.csv"
texts , labels = load_and_preprocess(df_path,
                                     text_column = "review",
                                     label_column = "sentiment",
                                     mode = "embedding")


In [ ]:
texts.head()

,review
0,one of the other reviewers has mentioned that ...
1,a wonderful little production the filming tech...
2,i thought this was a wonderful way to spend ti...
3,basically theres a family where a little boy j...
4,petter matteis love in the time of money is a ...


In [ ]:
labels

array([1, 1, 1, ..., 0, 0, 0])

In [ ]:
#build vocabulary
vocab = build_vocab(texts,max_vocab_size = 20_000)
#convert text to encoded sequence
sequences, tokenized_sentences = text_to_sequence(texts, vocab)
#Truncate extra length sequence and pad the shorter sequences
padded_seq = pad_sequence(sequences, seq_len= 250)


In [ ]:
embedding_size = 100
#word2vec model
# embedding_matrix = load_w2v(tokenized_sentences = tokenized_sentences, vocab = vocab, embedding_size = 100 )


In [ ]:
#Golve model
embedding_matrix = load_glove(vocab, embedding_size = embedding_size)
embedding_tensor = torch.tensor(embedding_matrix, dtype = torch.float32)

In [ ]:
#convert data to tensors
X = torch.tensor(padded_seq, dtype = torch.long)
y = torch.tensor(labels, dtype = torch.float32)

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3,random_state = 42)

In [ ]:
#create TensorDataset and DataLoaders
from torch.utils.data import TensorDataset, DataLoader

#create tensorDataset
train_set = TensorDataset(X_train, y_train)
test_set = TensorDataset(X_test, y_test)

#generate dataloaders
train_loader = DataLoader(train_set, batch_size = 64, shuffle = True)
test_loader = DataLoader(test_set, batch_size = 64)


In [ ]:
#cnn model
Sentiment_cnn = CNNModel(vocab_size = len(vocab), embedding_size = embedding_size, embedding_matrix = embedding_tensor)

model = Sentiment_cnn
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())


In [ ]:
#Train
start = time.time()
epochs = 15
for epoch in range(epochs):
  model.train()
  running_loss = 0.0
  for xb, yb in train_loader:
    #set the gradient to zero
    optimizer.zero_grad()
    #pass input to the model
    output = model(xb)
    #Apply sigmoid on squeezed output
    output = torch.sigmoid(output.squeeze())
    #Calculate the loss
    loss = criterion(output, yb)
    #Backward propagation
    loss.backward()
    #update the gradients
    optimizer.step()
    #update the loss
    running_loss+= loss.item()

  print(f"epoch {epoch+1}/{epochs} and loss {running_loss/len(train_loader)} ")
end = time.time()

epoch 1/15 and loss 0.39051773610237095 
epoch 2/15 and loss 0.2317014825028083 
epoch 3/15 and loss 0.1435247797525779 
epoch 4/15 and loss 0.06758034151639973 
epoch 5/15 and loss 0.024930131957922437 
epoch 6/15 and loss 0.00793827623145917 
epoch 7/15 and loss 0.0031669247335536404 
epoch 8/15 and loss 0.0016651673952647072 
epoch 9/15 and loss 0.0010012232690412802 
epoch 10/15 and loss 0.0006515163245048609 
epoch 11/15 and loss 0.0004364437420134836 
epoch 12/15 and loss 0.00029877447759412513 
epoch 13/15 and loss 0.00020789444622995717 
epoch 14/15 and loss 0.00014597460093976753 
epoch 15/15 and loss 0.00010328844078682181 


In [ ]:
correct_values = 0
total_values = 0
with torch.no_grad():
  model.eval()
  for xb, yb in test_loader:
    #pass input to the model
    output = model(xb)
    #Predicted Sentiment
    predicted = (torch.sigmoid(output.squeeze())>0.5).float()
    #update the correct predicted values
    correct_values += (predicted == yb).sum().item()
    #update the count of values in each batch
    total_values += yb.size(0)
  accuracy = correct_values/total_values*100
  print(f"Accuracy: {accuracy}")

Accuracy: 89.12666666666667


In [ ]:
results = {
    'Model' : "CNN(glove)",
    'vocab_size': len(vocab),
    'embedding_size': 100,
    'sequence_length': 250,
    'hidden_size': '-',
    'num_filters': 100,
    'num_layers' :1,
    'batch_size': 64,
    'num_epochs': 15,
    'Accuracy' :accuracy,
    'Time' : end-start
}

In [ ]:
new_df = pd.DataFrame([results])

In [ ]:
results_df = pd.concat([results_df, new_df], ignore_index = True)

In [ ]:
results_df = pd.read_csv("/content/drive/MyDrive/Sentiment_classification/results/pretrained_embedding_results.csv")

In [ ]:
results_df

,Model,vocab_size,embedding_size,sequence_length,hidden_size,num_filters,num_layers,batch_size,num_epochs,Accuracy,Time
0,CNN(Word2Vec),20002,100,250,-,100,1,64,15,89.600000,2836.796211
1,CNN(glove),20002,100,250,-,100,1,64,15,89.126667,2870.457033


In [3]:
!git status

Refresh index: 100% (16/16), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    experiments/01_tfidf_RNN.ipynb
	modified:   experiments/02_embedding_nn.ipynb
	modified:   results/tfidf_results.csv
	modified:   src/__pycache__/encoded_tokens.cpython-312.pyc
	modified:   src/__pycache__/models.cpython-312.pyc
	modified:   src/__pycache__/preprocess.cpython-312.pyc
	modified:   src/encoded_tokens.py
	modified:   src/models.py
	deleted:    src/train.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/glove.6B.100d.txt
	experiments/01_tfidf_with_linear_models.ipynb
	experiments/03_pretrained_embedding.ipynb
	final_models_comapar.png
	requirements.txt
	results/pretrained_embedding_results.csv
	results/results_plots.ipynb
	seq_vs_emb.png
	src/__pycache__/pretr

In [15]:
%%writefile /content/drive/MyDrive/Sentiment_classification/.gitignore
# Ignore python cache
**/__pycache__/
*.pyc

# Ignore large datasets and embeddings
data/

# Ignore local plots
*.png

Overwriting /content/drive/MyDrive/Sentiment_classification/.gitignore


In [6]:
!git rm -r --cached data/

rm 'data/IMDB Dataset.csv'


In [10]:
!git commit -m "Remove data folder form version control"

[main b284684] Remove data folder form version control
 1 file changed, 50001 deletions(-)
 delete mode 100644 data/IMDB Dataset.csv


In [9]:
!git config --global user.email "vsravani2024@gmail.com"
!git config --global user.name "Sravani-AIML"

In [13]:
!git push origin main

To https://github.com/Sravani-AIML/Sentiment_classification.git
 ! [rejected]        main -> main (non-fast-forward)
error: failed to push some refs to 'https://github.com/Sravani-AIML/Sentiment_classification.git'
hint: Updates were rejected because the tip of your current branch is behind
hint: its remote counterpart. Integrate the remote changes (e.g.
hint: 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [14]:
!git pull origin main
Transfer Learning with pretrained embedding models
Word2Vec outperformed the pre-trained GloVe 100d because it was trained directly on the movie review dataset, allowing it to capture domain-specific sentiment and vocabulary nuances. In contrast, the GloVe embeddings relied on global co-occurrence statistics from general corpora like Wikipedia, which lacked the specialized context necessary for high-accuracy sentiment classification in this specific domain.

From https://github.com/Sravani-AIML/Sentiment_classification
 * branch            main       -> FETCH_HEAD
hint: You have divergent branches and need to specify how to reconcile them.
hint: You can do so by running one of the following commands sometime before
hint: your next pull:
hint: 
hint:   git config pull.rebase false  # merge (the default strategy)
hint:   git config pull.rebase true   # rebase
hint:   git config pull.ff only       # fast-forward only
hint: 
hint: You can replace "git config" with "git config --global" to set a default
hint: preference for all repositories. You can also pass --rebase, --no-rebase,
hint: or --ff-only on the command line to override the configured default per
hint: invocation.
fatal: Need to specify how to reconcile divergent branches.


In [16]:
!git add src/encoded_tokens.py src/models.py src/pretrained_embeddings.py src/preprocess.py

In [17]:
!git add experiments/01_tfidf_with_linear_models.ipynb experiments/02_embedding_nn.ipynb experiments/03_pretrained_embedding.ipynb

In [19]:
!git add requirements.txt experiments/notes.md